# Cross-Dataset Validation — MIT-BIH AFDB Model → 12-Lead ECG Arrhythmia Database

**Goal:** Test the model trained on MIT-BIH AFDB on a completely different dataset to evaluate generalization.

**Training data:** MIT-BIH AFDB (2 leads, 250 Hz, 10-hour ambulatory recordings, 23 patients)  
**Test data:** Large Scale 12-Lead ECG Arrhythmia Database (12 leads → use Lead I + Lead II, 500 Hz → downsample to 250 Hz, 10-second recordings, ~45,000 patients)

### Key differences between datasets:
- **Recording length:** 10 hours (AFDB) vs 10 seconds (arrhythmia DB)
- **Sampling rate:** 250 Hz vs 500 Hz (will downsample)
- **Population:** 23 patients vs ~45,000 patients
- **Label type:** Per-sample rhythm annotations vs per-recording diagnosis codes (SNOMED-CT)
- **AFib code:** `164889003` = "ECG: atrial fibrillation"


## 1. Imports

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, classification_report, matthews_corrcoef,
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    brier_score_loss
)
from sklearn.calibration import calibration_curve
from scipy.signal import resample
import time
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
import wfdb
import glob

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## 2. Model Definition (Must match training architecture)

In [2]:
class KanResInit(nn.Module):
    def __init__(self, in_channels, filterno_1, filterno_2, filtersize_1, filtersize_2, stride):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, filterno_1, filtersize_1, stride=stride)
        self.bn1 = nn.BatchNorm1d(filterno_1)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv1d(filterno_1, filterno_2, filtersize_2)
        self.bn2 = nn.BatchNorm1d(filterno_2)
        self.relu2 = nn.ReLU()

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        return x

class KanResModule(nn.Module):
    def __init__(self, in_channels, filterno_1, filterno_2, filtersize_1, filtersize_2, stride):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, filterno_1, filtersize_1, stride=stride, padding='same')
        self.bn1 = nn.BatchNorm1d(filterno_1)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv1d(filterno_1, filterno_2, filtersize_2, padding='same')
        self.bn2 = nn.BatchNorm1d(filterno_2)
        self.relu2 = nn.ReLU()

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu2(out)
        out = out + identity
        return out

class KanResWideX(nn.Module):
    def __init__(self, input_channels=2, output_size=2):
        super().__init__()
        self.init_block = KanResInit(input_channels, 64, 32, 8, 3, 1)
        self.pool = nn.AvgPool1d(kernel_size=2)
        self.res_modules = nn.ModuleList([
            KanResModule(32, 64, 32, 50, 50, 1) for _ in range(8)
        ])
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(32, output_size)

    def forward(self, x):
        x = self.init_block(x)
        x = self.pool(x)
        for res_module in self.res_modules:
            x = res_module(x)
        x = self.global_pool(x)
        x = x.squeeze(-1)
        x = self.fc(x)
        return x

print("Model defined.")

Model defined.


## 3. Configuration

In [3]:
class Config:
    # Path to the 12-lead arrhythmia database
    arrhythmia_dir = r"C:/Users/Admin/OneDrive/Skrivebord/Afib-Master/data/ecg-arrhythmia/a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0"
    wfdb_dir = os.path.join(arrhythmia_dir, "WFDBRecords")

    # Path to trained model (UPDATE THIS after training your 5-fold CV model)
    # Save your best model with: torch.save(model.state_dict(), "best_model_2ch.pth")
    model_path = r"C:/Users/Admin/OneDrive/Skrivebord/Afib-Master/best_model_2ch.pth"

    # Signal parameters
    source_fs = 500       # Arrhythmia database sampling rate
    target_fs = 250       # MIT-BIH AFDB sampling rate (what model expects)
    num_channels = 2      # Lead I (index 0) + Lead II (index 1)
    lead_indices = [0, 1] # Channel indices for Lead I and Lead II

    # Windowing (must match training)
    window_seconds = 4
    window_size = target_fs * window_seconds  # 1000 samples at 250 Hz

    # AFib SNOMED-CT code
    afib_code = "164889003"

    # Inference
    batch_size = 128
    num_classes = 2

config = Config()
assert os.path.exists(config.wfdb_dir), f"WFDBRecords not found: {config.wfdb_dir}"
print(f"Arrhythmia DB: {config.arrhythmia_dir}")
print(f"Source fs: {config.source_fs} Hz → Target fs: {config.target_fs} Hz")
print(f"Leads: I (idx 0) + II (idx 1)")
print(f"Window: {config.window_size} samples ({config.window_seconds}s)")
print(f"AFib SNOMED code: {config.afib_code}")

Arrhythmia DB: C:/Users/Admin/OneDrive/Skrivebord/Afib-Master/data/ecg-arrhythmia/a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0
Source fs: 500 Hz → Target fs: 250 Hz
Leads: I (idx 0) + II (idx 1)
Window: 1000 samples (4s)
AFib SNOMED code: 164889003


## 4. Discover All Records

The database has a nested folder structure: `WFDBRecords/XX/XXX/JSXXXXX`  
We find all `.hea` files recursively.

In [4]:
# Find all .hea files recursively
hea_files = glob.glob(os.path.join(config.wfdb_dir, "**", "*.hea"), recursive=True)
print(f"Found {len(hea_files)} records")

# Show a few examples
for f in hea_files[:5]:
    print(f"  {f}")

Found 45152 records
  C:/Users/Admin/OneDrive/Skrivebord/Afib-Master/data/ecg-arrhythmia/a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0\WFDBRecords\01\010\JS00001.hea
  C:/Users/Admin/OneDrive/Skrivebord/Afib-Master/data/ecg-arrhythmia/a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0\WFDBRecords\01\010\JS00002.hea
  C:/Users/Admin/OneDrive/Skrivebord/Afib-Master/data/ecg-arrhythmia/a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0\WFDBRecords\01\010\JS00004.hea
  C:/Users/Admin/OneDrive/Skrivebord/Afib-Master/data/ecg-arrhythmia/a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0\WFDBRecords\01\010\JS00005.hea
  C:/Users/Admin/OneDrive/Skrivebord/Afib-Master/data/ecg-arrhythmia/a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0\WFDBRecords\01\010\JS00006.hea


## 5. Parse Labels from Header Files

Each `.hea` file has a `#Dx:` line with SNOMED-CT codes.  
A record is AFib if `164889003` is among its codes, otherwise Non-AFib.

In [5]:
def parse_diagnosis_from_hea(hea_path):
    """Read the #Dx line from a .hea file and return list of SNOMED codes."""
    codes = []
    with open(hea_path, 'r') as f:
        for line in f:
            if line.startswith('#Dx:'):
                code_str = line.strip().split(':')[1].strip()
                codes = [c.strip() for c in code_str.split(',')]
                break
    return codes

# Parse all records
records = []
afib_count = 0
non_afib_count = 0

for hea_path in hea_files:
    codes = parse_diagnosis_from_hea(hea_path)
    is_afib = config.afib_code in codes
    record_dir = os.path.dirname(hea_path)
    record_name = os.path.splitext(os.path.basename(hea_path))[0]
    records.append({
        "path": os.path.join(record_dir, record_name),
        "name": record_name,
        "codes": codes,
        "is_afib": is_afib,
        "label": 1 if is_afib else 0
    })
    if is_afib:
        afib_count += 1
    else:
        non_afib_count += 1

print(f"Total records: {len(records)}")
print(f"  AFib:     {afib_count} ({afib_count/len(records)*100:.1f}%)")
print(f"  Non-AFib: {non_afib_count} ({non_afib_count/len(records)*100:.1f}%)")

KeyboardInterrupt: 

## 6. Load and Preprocess Signals

For each record:
1. Read Lead I and Lead II from the 12-lead recording
2. Downsample from 500 Hz to 250 Hz
3. Split into 4-second windows (no overlap for test data)
4. Z-normalize each window per channel

In [ ]:
def load_and_preprocess_record(record_info, config):
    """Load a 12-lead record, extract Lead I + II, downsample, and create windows."""
    try:
        rec = wfdb.rdrecord(record_info["path"])
        signal = rec.p_signal  # (n_samples, 12)
    except Exception as e:
        return None, None

    # Extract Lead I and Lead II
    lead_signal = signal[:, config.lead_indices]  # (n_samples, 2)

    # Check for NaN
    if np.any(np.isnan(lead_signal)):
        return None, None

    # Downsample from 500 Hz to 250 Hz
    n_samples_original = lead_signal.shape[0]
    n_samples_target = int(n_samples_original * config.target_fs / config.source_fs)
    lead_resampled = np.zeros((n_samples_target, 2))
    for ch in range(2):
        lead_resampled[:, ch] = resample(lead_signal[:, ch], n_samples_target)

    # Create windows (no overlap for test — use full window stride)
    windows = []
    win_start = 0
    while win_start + config.window_size <= n_samples_target:
        win_signal = lead_resampled[win_start:win_start + config.window_size, :].copy()

        # Z-normalize per channel
        valid = True
        for ch in range(2):
            std = np.std(win_signal[:, ch])
            if std < 1e-6:
                valid = False
                break
            win_signal[:, ch] = (win_signal[:, ch] - np.mean(win_signal[:, ch])) / std

        if valid:
            windows.append(win_signal.T.copy())  # (2, 1000)

        win_start += config.window_size  # No overlap

    if not windows:
        return None, None

    windows = np.array(windows, dtype=np.float32)
    labels = np.full(len(windows), record_info["label"], dtype=np.int64)
    return windows, labels

# Test on one record
test_rec = records[0]
w, l = load_and_preprocess_record(test_rec, config)
if w is not None:
    print(f"Test record {test_rec['name']}: {w.shape} windows, label={test_rec['label']} ({'AFib' if test_rec['label']==1 else 'Non-AFib'})")
    print(f"  Original: 5000 samples at 500 Hz (10s)")
    print(f"  Resampled: 2500 samples at 250 Hz (10s)")
    print(f"  Windows: {w.shape[0]} x {w.shape[2]} samples = {w.shape[0]} x {config.window_seconds}s")
else:
    print("Failed to load test record")

## 7. Load All Records

In [ ]:
print(f"Loading {len(records)} records...")
start_time = time.time()

all_windows = []
all_labels = []
skipped = 0
records_loaded = {"afib": 0, "non_afib": 0}

for i, rec in enumerate(records):
    if (i + 1) % 5000 == 0:
        elapsed = time.time() - start_time
        print(f"  Processed {i+1}/{len(records)} ({elapsed:.0f}s)...")

    windows, labels = load_and_preprocess_record(rec, config)
    if windows is not None:
        all_windows.append(windows)
        all_labels.append(labels)
        if rec["label"] == 1:
            records_loaded["afib"] += 1
        else:
            records_loaded["non_afib"] += 1
    else:
        skipped += 1

all_windows = np.concatenate(all_windows, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

load_time = time.time() - start_time
print(f"\nLoading complete in {load_time/60:.1f} minutes")
print(f"Records loaded: {records_loaded['afib'] + records_loaded['non_afib']} (skipped: {skipped})")
print(f"  AFib records:     {records_loaded['afib']}")
print(f"  Non-AFib records: {records_loaded['non_afib']}")
print(f"\nTotal windows: {len(all_labels):,}")
print(f"  AFib windows:     {np.sum(all_labels == 1):,}")
print(f"  Non-AFib windows: {np.sum(all_labels == 0):,}")
print(f"  AFib prevalence:  {np.mean(all_labels):.2%}")

## 8. Load Trained Model

**IMPORTANT:** You need to save your trained model first!  
In your training notebook, after training, add:
```python
torch.save(best_model_state, "best_model_2ch.pth")
```
Then update `config.model_path` to point to the saved file.

In [ ]:
# Load the trained model
model = KanResWideX(input_channels=config.num_channels, output_size=config.num_classes).to(device)

if os.path.exists(config.model_path):
    state_dict = torch.load(config.model_path, map_location=device)
    model.load_state_dict(state_dict)
    print(f"Model loaded from: {config.model_path}")
else:
    print(f"WARNING: Model file not found at {config.model_path}")
    print("Please save your trained model first and update the path.")
    print("In your training notebook, add:")
    print('  torch.save(best_model_state, "best_model_2ch.pth")')

model.eval()
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

## 9. Run Inference

In [ ]:
class ECGDataset(Dataset):
    def __init__(self, windows, labels):
        self.windows = torch.FloatTensor(windows)
        self.labels = torch.LongTensor(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.windows[idx], self.labels[idx]

test_loader = DataLoader(ECGDataset(all_windows, all_labels),
                         batch_size=config.batch_size, shuffle=False,
                         num_workers=0, pin_memory=True)

print(f"Running inference on {len(all_labels):,} windows...")
start_time = time.time()

all_preds = []
all_targets = []
all_probs = []

model.eval()
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(targets.numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

all_targets = np.array(all_targets)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

inference_time = time.time() - start_time
print(f"Inference complete in {inference_time:.1f} seconds")

## 10. Compute All Metrics

In [ ]:
cm = confusion_matrix(all_targets, all_preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

acc = accuracy_score(all_targets, all_preds)
prec = precision_score(all_targets, all_preds, pos_label=1, zero_division=0)
rec = recall_score(all_targets, all_preds, pos_label=1, zero_division=0)
f1 = f1_score(all_targets, all_preds, pos_label=1, zero_division=0)
mcc = matthews_corrcoef(all_targets, all_preds)
auroc = roc_auc_score(all_targets, all_probs)
auprc = average_precision_score(all_targets, all_probs)
brier = brier_score_loss(all_targets, all_probs)

# ECE and MCE
n_bins = 10
bin_boundaries = np.linspace(0, 1, n_bins + 1)
ece, mce = 0.0, 0.0
for i in range(n_bins):
    if i < n_bins - 1:
        mask = (all_probs >= bin_boundaries[i]) & (all_probs < bin_boundaries[i + 1])
    else:
        mask = (all_probs >= bin_boundaries[i]) & (all_probs <= bin_boundaries[i + 1])
    bin_count = mask.sum()
    if bin_count > 0:
        bin_acc = all_targets[mask].mean()
        bin_conf = all_probs[mask].mean()
        cal_error = abs(bin_acc - bin_conf)
        ece += (bin_count / len(all_targets)) * cal_error
        mce = max(mce, cal_error)

print("=" * 65)
print("CROSS-DATASET VALIDATION RESULTS")
print("Trained on: MIT-BIH AFDB | Tested on: 12-Lead ECG Arrhythmia DB")
print("=" * 65)
print(f"  Accuracy:   {acc:.4f}")
print(f"  Precision:  {prec:.4f}")
print(f"  Recall:     {rec:.4f}")
print(f"  F1:         {f1:.4f}")
print(f"  MCC:        {mcc:.4f}")
print(f"  AUROC:      {auroc:.4f}")
print(f"  AUPRC:      {auprc:.4f}")
print(f"  Brier:      {brier:.4f}")
print(f"  ECE:        {ece:.4f}")
print(f"  MCE:        {mce:.4f}")
print(f"  TN={tn}  FP={fp}  FN={fn}  TP={tp}")
print()
print(classification_report(all_targets, all_preds, target_names=["Non-AFib", "AFib"]))

## 11. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges",
            xticklabels=["Non-AFib", "AFib"], yticklabels=["Non-AFib", "AFib"],
            ax=ax, annot_kws={"size": 14})
ax.set_title(f"Cross-Dataset Confusion Matrix\nAcc={acc:.3f} F1={f1:.3f} MCC={mcc:.3f}", fontsize=13, fontweight='bold')
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## 12. ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
fpr, tpr, _ = roc_curve(all_targets, all_probs)
ax.plot(fpr, tpr, color='red', linewidth=2.5, label=f'Cross-Dataset (AUROC={auroc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Cross-Dataset Validation', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Precision-Recall Curve

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
prec_c, rec_c, _ = precision_recall_curve(all_targets, all_probs)
ax.plot(rec_c, prec_c, color='red', linewidth=2.5, label=f'Cross-Dataset (AUPRC={auprc:.3f})')
baseline = np.mean(all_targets)
ax.axhline(y=baseline, color='k', linestyle='--', alpha=0.5, label=f'Baseline (prevalence={baseline:.2f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve — Cross-Dataset Validation', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.show()

## 14. Calibration Plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
fraction_pos, mean_predicted = calibration_curve(all_targets, all_probs, n_bins=10, strategy='uniform')
ax.plot(mean_predicted, fraction_pos, 's-', color='red', linewidth=2, markersize=6, label='Model')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect')
ax2 = ax.twinx()
ax2.hist(all_probs, bins=20, range=(0, 1), alpha=0.15, color='red')
ax2.set_ylabel('Count', fontsize=9, alpha=0.5)
ax2.tick_params(axis='y', labelsize=8, colors='gray')
ax.set_title(f"Calibration — Cross-Dataset\nECE={ece:.4f}  MCE={mce:.4f}  Brier={brier:.4f}", fontsize=13, fontweight='bold')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.legend(loc='upper left', fontsize=10)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 15. Predicted Probability Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
mask_neg = all_targets == 0
mask_pos = all_targets == 1
ax.hist(all_probs[mask_neg], bins=50, range=(0, 1), alpha=0.6, color='green', label='Non-AFib', density=True)
ax.hist(all_probs[mask_pos], bins=50, range=(0, 1), alpha=0.6, color='red', label='AFib', density=True)
ax.axvline(x=0.5, color='black', linestyle='--', alpha=0.5)
ax.set_title("Predicted Probability Distribution — Cross-Dataset", fontsize=13, fontweight='bold')
ax.set_xlabel("P(AFib)"); ax.set_ylabel("Density")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 16. Final Summary

In [ ]:
print("=" * 70)
print("CROSS-DATASET VALIDATION SUMMARY")
print("=" * 70)
print()
print("Training dataset:  MIT-BIH Atrial Fibrillation Database")
print("  - 23 patients, 2 leads, 250 Hz, 10-hour recordings")
print("  - Per-sample rhythm annotations")
print()
print("Testing dataset:   Large Scale 12-Lead ECG Arrhythmia Database")
print(f"  - {records_loaded['afib'] + records_loaded['non_afib']:,} patients, 12 leads (used I + II), 500 Hz → 250 Hz")
print("  - 10-second recordings, per-recording SNOMED-CT diagnosis")
print(f"  - AFib records: {records_loaded['afib']:,}, Non-AFib records: {records_loaded['non_afib']:,}")
print()
print(f"Total windows evaluated: {len(all_labels):,}")
print(f"  AFib:     {np.sum(all_labels == 1):,} ({np.mean(all_labels):.2%})")
print(f"  Non-AFib: {np.sum(all_labels == 0):,} ({1 - np.mean(all_labels):.2%})")
print()
print(f"{'Metric':<15} {'Value':<10}")
print("-" * 25)
for name, val in [("Accuracy", acc), ("Precision", prec), ("Recall", rec),
                   ("F1", f1), ("MCC", mcc), ("AUROC", auroc), ("AUPRC", auprc),
                   ("Brier", brier), ("ECE", ece), ("MCE", mce)]:
    print(f"{name:<15} {val:.4f}")
print()
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")
print()
print(f"Inference time: {inference_time:.1f} seconds")